Installing dependencies

In [1]:
# import warnings
# warnings.simplefilter('ignore')
!pip install langchain_community
# !pip install unstructured[docx]
# !pip install unstructured[pdf]
!pip install accelerate
!pip install langchain
# !pip install pypdf
!pip install -U bitsandbytes
# !pip install -U git+https://github.com/huggingface/peft.git
# !pip install -U git+https://github.com/huggingface/accelerate.git
!pip install -U einops
!pip install -U safetensors
!pip install -U xformers
!pip install -U ctransformers[cuda]
# !pip install huggingface_hub
!pip install chromadb
!pip install sentence-transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 32.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 973.5/973.5 kB 60.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.5/308.5 kB 35.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.8/122.8 kB 18.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.5/142.5 kB 17.7 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 24.0
    Uninstalling packaging-24.0:
      Successfully uninstalled packaging-24.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.6/302.6 kB 7.1 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using c

Initializing variables

In [2]:
CHROMA_PATH = '/content/drive/MyDrive/juris/chroma'
DATA_PATH = "/content/drive/MyDrive/rules"
EMBEDDING_PATH = '/content/drive/MyDrive/juris/multilingual-e5-large'
MODEL_DRIVE = "/content/drive/MyDrive/juris/saiga_llama3_8b"
CHUNK_SIZE = 750
CHUNK_OVERLAP = 200

Initializing dependencies

In [3]:
# from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain.llms import HuggingFacePipeline
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
import transformers
import torch
import sys
from torch import cuda, bfloat16
import transformers
from langchain.vectorstores import Chroma
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
# Define the model directory
model_directory = 'https://huggingface.co/microsoft/Phi-3-medium-128k-instruct'
# Load the tokenizer and the model



Retrieving vector store

In [4]:
# Initialize embedding function and vector store
embedding_function = SentenceTransformerEmbeddings(model_name=EMBEDDING_PATH)
vectorstore = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_function)
retriever = vectorstore.as_retriever(search_type='similarity', k=100)

Retrieving model

In [ ]:
torch.random.manual_seed(0)
model_id = "microsoft/Phi-3-medium-128k-instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

pipeline = pipeline("text-generation",
                    model=model,
                    tokenizer=tokenizer,
                    torch_dtype=torch.float16,
                    max_length=8000,
                    max_new_tokens=8000,
                    do_sample=True,
                    temperature=0.5,
                    top_p=0.4,
                    repetition_penalty=1.1,
                    device_map="auto")
llm = HuggingFacePipeline(pipeline=pipeline)

In [ ]:
# from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
# from langchain.vectorstores import Chroma

# # Initialize embedding function and vector store
# embedding_function = SentenceTransformerEmbeddings(model_name=EMBEDDING_PATH)
# vectorstore = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_function)
# retriever = vectorstore.as_retriever(search_type='similarity', k=100)



# Usage example:
# pipeline = Pipeline(llm_instance, retriever)
# result = pipeline.generate("What are the regulations for child labor?")
# print(result)

# Usage example:
# pipeline = Pipeline(llm_instance, retriever)
# result = pipeline.generate("What are the regulations for child labor?")
# print(result)

class Pipeline:
    def __init__(self, llm, retriever):
        self.llm = llm
        self.retriever = retriever

    def retrieve(self, data):
        docs = self.retriever.invoke(data)
        print('here')
        # Combine content and metadata for each document
        combined_docs = []
        for doc in docs:
            content = doc.page_content
            file_name = doc.metadata.get("file_name", "Неизвестно")
            combined_docs.append(f"Ресурс: {file_name}\Контекст: {content}")
        return "\n\n".join(combined_docs)
    def augment(self,question,context):
        return f"""Ответь на вопрос: '{question}' базируясь и проанализировав только на предоставленном контексте кодекса Республики Казахстан. В конце предоставь ссылки, статьи, ресурсы статей на которых основывается вывод. Это контекст:
        {context};
        """
    # def parse(self, string):
    #     # Extract the text after "Вывод: "
    #     try:
    #         answer_start = string.index("Вывод: ") + len("Вывод: ")
    #         answer = string[answer_start:].strip()
    #         return answer
    #     except ValueError:
    #         # If "Вывод: " is not found, return an appropriate message or the original string
    #         return "Вывод не найден в ответе модели."

    def generate(self, question):
        context = self.retrieve(question)
        # print(context)
        prompt = self.augment(question, context)
        print(prompt)
        answer = self.llm.invoke(prompt)
        return answer
# class Pipeline:
#     def __init__(self, llm, retriever):
#         self.llm = llm
#         self.retriever = retriever

#     def retrieve(self, data):
#         docs = self.retriever.invoke(data[0])
#         print('here')
#         # Combine content and metadata for each document
#         combined_docs = []
#         for doc in docs:
#             content = doc.page_content
#             file_name = doc.metadata.get("file_name", "Неизвестно")
#             combined_docs.append(f"Ресурс: {file_name}\Контекст: {content}")
#         return "\n\n".join(combined_docs)

#     def augment(self, data, context):
#         if data[2] == True:
#             return f"""
#             Это предыдущий ответ: {data[1]}.
#             Ниже тебе предоставлен контекст по вопросу '{data[0]}': {context}.
#             Cоставь текст ответ на основе предоставленного контекста.
#             Также предоставь в конце ресурсы на которых базируется ответ: статьи, ресурсы от статей, и если есть ссылки, предоставь ссылки.
#             Если контекст не даёт ответа, отправь 'Не знаю'.
#         """
#         return f"""
#         Ниже тебе предоставлен контекст по вопросу '{data[0]}': {context}.
#         Cоставь текст ответ на основе предоставленного контекста.
#         Также предоставь в конце ресурсы на которых базируется ответ: статьи, ресурсы от статей, и если есть ссылки, предоставь ссылки.
#         Если контекст не даёт ответа, отправь 'Не знаю'.
#         """

#     def generate(self, question):
#         # data[0] = data[0] + ' ' + 'с статьями и ссылками'
#         context = self.retrieve(question)
#         prompt = self.augment(question, context)
#         print(prompt)
#         answer = self.llm.invoke(prompt)
#         return answer
def llama3_chat():
    print("Привет, я ИИ который может помочь с вопросами связанные с бюрографией. \nПожалуйста, как только вы удовлетворены ответом, напишите СТОП")
    print()
    print("-------------------------------------------------------------------------------------")
    pipe = Pipeline(llm,retriever)
    question = input()
    # data = []
    # data.append(question)
    # data.append('Здесь ничего нету')
    # data.append(False)
    while question != "СТОП":
        # if question.strip().startswith("продолжить"):
        #   data[0]=question
        #   data[1]=out
        #   data[2]=True
        out  = pipe.generate(question)
        # print("\nУдовлетворяет ли вас ответ? Если хотите продолжить с последнего места, напишите 'продолжить, ВАШ ВОПРОС)', либо задайте другой вопрос, или СТОП")
        print('Вы можете остановить модель, написав СТОП')
        print("-------------------------------------------------------------------------------------")
        question = input()
llama3_chat()

In [ ]:
# class Pipeline:
#     def __init__(self, llm, retriever):
#         self.llm = llm
#         self.retriever = retriever

#     def retrieve(self, question):
#         docs = self.retriever.invoke(question)
#         # Combine content and metadata for each document
#         combined_docs = []
#         for doc in docs:
#             content = doc.page_content
#             file_name = doc.metadata.get("file_name", "Неизвестно")
#             combined_docs.append(f"Ресурс: {file_name}\Контекст: {content}")
#         return "\n\n".join(combined_docs)

#     def augment(self, data, context):
#         if data[2] == True:
#             return f"""
#         Это твои предыдущий ответ: {data[1]}.
#         Ниже тебе предоставлен контекст по вопросу '{data[0]}': {context}.
#         Cоставь текст ответ на основе предоставленного контекста.
#         Также предоставь в конце ресурсы на которых базируется ответ: статьи, ресурсы от статей, и если есть ссылки, предоставь ссылки.
#         Если контекст не даёт ответа, отправь 'Не знаю'.
#         """
#         return f"""
#         Ниже тебе предоставлен контекст по вопросу '{data[0]}': {context}.
#         Cоставь текст ответ на основе предоставленного контекста.
#         Также предоставь в конце ресурсы на которых базируется ответ: статьи, ресурсы от статей, и если есть ссылки, предоставь ссылки.
#         Если контекст не даёт ответа, отправь 'Не знаю'.
#         """

#     def parse(self, string):
#         # Extract the text after "Вывод: "
#         try:
#             answer_start = string.index("Результат:") + len("Результат:")
#             answer = string[answer_start:].strip()
#             return answer
#         except ValueError:
#             # If "Вывод: " is not found, return an appropriate message or the original string
#             return "Вывод не найден в ответе модели."

#     def generate(self, data):
#         data[0] = data[0] + ' ' + 'с статьями и ссылками'
#         context = self.retrieve(data[0])
#         prompt = self.augment(data, context)
#         print(prompt)
#         answer = self.llm.invoke(prompt)
#         return self.parse(answer)

